# trackpy 3D locate Benchmark (Colab)

このノートブックは `trackpy.locate` の以下を比較します。
- 実行時間 (`python / numba / cupy`)
- 検出結果の一致度（特徴点数、マッチ率、座標差）

## 使い方
1. Colab のランタイムを **GPU** に変更
2. このノートブックを上から順に実行
3. `Runtime` と `Detection` の2つの表を確認


In [ ]:
!python --version
!nvidia-smi || true

In [ ]:
# If you uploaded your modified repo to /content/trackpy-master,
# this cell installs that exact code. Otherwise it falls back to PyPI.
import os

repo_path = "/content/trackpy-master"
if os.path.isdir(repo_path):
    print(f"Installing local repo: {repo_path}")
    !pip -q install -U pip setuptools wheel
    !pip -q install -e "$repo_path"
else:
    print("Local repo not found. Installing trackpy from PyPI.")
    !pip -q install -U pip setuptools wheel
    !pip -q install trackpy

# numba + cupy (CUDA12). If Colab image changes, switch cupy build accordingly.
!pip -q install numba cupy-cuda12x

In [ ]:
import time
import numpy as np
import pandas as pd
import trackpy as tp
from scipy.spatial import cKDTree

try:
    import numba
except Exception:
    numba = None

try:
    import cupy as cp
except Exception:
    cp = None

print("trackpy:", tp.__version__)
print("numba:", None if numba is None else numba.__version__)
print("cupy:", None if cp is None else cp.__version__)


In [ ]:
from trackpy.artificial import draw_spots, gen_nonoverlapping_locations

def make_image(shape=(128, 128, 64), count=200, diameter=(9, 9, 9), noise_level=20, seed=42):
    rng = np.random.default_rng(seed)
    radius = tuple(d // 2 for d in diameter)
    margin = tuple(r + 1 for r in radius)
    separation = tuple(d * 3 for d in diameter)
    positions = gen_nonoverlapping_locations(shape, count, separation, margin)
    positions = positions + rng.random(positions.shape) * 0.2
    size = [d / 2 for d in diameter]
    image = draw_spots(shape, positions, size, noise_level)
    return image.astype(np.uint16, copy=False)

def run_once(image, diameter, engine):
    t0 = time.perf_counter()
    f = tp.locate(image, diameter, engine=engine)
    ms = (time.perf_counter() - t0) * 1000.0
    return ms, f

def bench_engine(image, diameter, engine, warmup=2, repeats=5):
    for _ in range(warmup):
        _ = run_once(image, diameter, engine)

    times = []
    counts = []
    for _ in range(repeats):
        ms, f = run_once(image, diameter, engine)
        times.append(ms)
        counts.append(len(f))

    return {
        "engine": engine,
        "mean_ms": float(np.mean(times)),
        "min_ms": float(np.min(times)),
        "max_ms": float(np.max(times)),
        "std_ms": float(np.std(times)),
        "features": int(np.mean(counts)),
        "status": "ok",
    }

def benchmark_all(shape=(128, 128, 64), count=200, diameter=(9, 9, 9), noise_level=20, seed=42,
                  engines=("python", "numba", "cupy"), warmup=2, repeats=5):
    image = make_image(shape=shape, count=count, diameter=diameter, noise_level=noise_level, seed=seed)
    rows = []
    for engine in engines:
        try:
            rows.append(bench_engine(image, diameter, engine, warmup=warmup, repeats=repeats))
        except Exception as exc:
            rows.append({
                "engine": engine,
                "mean_ms": np.nan,
                "min_ms": np.nan,
                "max_ms": np.nan,
                "std_ms": np.nan,
                "features": np.nan,
                "status": f"ERROR: {type(exc).__name__}: {exc}",
            })

    df = pd.DataFrame(rows)
    base = df.loc[df["engine"] == "python", "mean_ms"]
    if len(base) and np.isfinite(base.iloc[0]):
        df["speedup_vs_python"] = base.iloc[0] / df["mean_ms"]
    else:
        df["speedup_vs_python"] = np.nan
    return df

def _extract_xyz(df):
    if df is None or len(df) == 0:
        return np.empty((0, 3), dtype=float)
    cols = ["z", "y", "x"]
    return df[cols].to_numpy(dtype=float)

def _compare_positions(ref_xyz, tgt_xyz, tol=1.0):
    if len(ref_xyz) == 0 or len(tgt_xyz) == 0:
        return {
            "matched": 0,
            "match_ratio_ref": 0.0,
            "rmse_xyz": np.nan,
            "mae_z": np.nan,
            "mae_y": np.nan,
            "mae_x": np.nan,
        }

    tree = cKDTree(tgt_xyz)
    dist, idx = tree.query(ref_xyz, k=1)
    mask = np.isfinite(dist) & (dist <= tol)
    matched = int(mask.sum())
    if matched == 0:
        return {
            "matched": 0,
            "match_ratio_ref": 0.0,
            "rmse_xyz": np.nan,
            "mae_z": np.nan,
            "mae_y": np.nan,
            "mae_x": np.nan,
        }

    delta = ref_xyz[mask] - tgt_xyz[idx[mask]]
    rmse = float(np.sqrt(np.mean(np.sum(delta ** 2, axis=1))))
    mae = np.mean(np.abs(delta), axis=0)
    return {
        "matched": matched,
        "match_ratio_ref": matched / max(len(ref_xyz), 1),
        "rmse_xyz": rmse,
        "mae_z": float(mae[0]),
        "mae_y": float(mae[1]),
        "mae_x": float(mae[2]),
    }

def compare_detection_all(shape=(128, 128, 64), count=200, diameter=(9, 9, 9), noise_level=20, seed=42,
                          engines=("python", "numba", "cupy"), ref_engine="python", tol=1.0):
    image = make_image(shape=shape, count=count, diameter=diameter, noise_level=noise_level, seed=seed)
    detections = {}
    statuses = {}
    for engine in engines:
        try:
            detections[engine] = tp.locate(image, diameter, engine=engine)
            statuses[engine] = "ok"
        except Exception as exc:
            detections[engine] = None
            statuses[engine] = f"ERROR: {type(exc).__name__}: {exc}"

    ref_df = detections.get(ref_engine)
    ref_xyz = _extract_xyz(ref_df)
    rows = []
    for engine in engines:
        row = {
            "engine": engine,
            "status": statuses[engine],
            "ref_engine": ref_engine,
            "ref_features": len(ref_df) if ref_df is not None else np.nan,
            "target_features": len(detections[engine]) if detections[engine] is not None else np.nan,
        }

        if statuses[engine] == "ok" and ref_df is not None:
            tgt_xyz = _extract_xyz(detections[engine])
            row.update(_compare_positions(ref_xyz, tgt_xyz, tol=tol))
        else:
            row.update({
                "matched": np.nan,
                "match_ratio_ref": np.nan,
                "rmse_xyz": np.nan,
                "mae_z": np.nan,
                "mae_y": np.nan,
                "mae_x": np.nan,
            })

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
# Runtime benchmark (isotropic 3D)
df_runtime_iso = benchmark_all(
    shape=(128, 128, 64),
    count=200,
    diameter=(9, 9, 9),
    noise_level=20,
    seed=42,
    engines=("python", "numba", "cupy"),
    warmup=2,
    repeats=5,
)
df_runtime_iso.sort_values("engine")

In [ ]:
# Detection comparison (isotropic 3D)
df_detect_iso = compare_detection_all(
    shape=(128, 128, 64),
    count=200,
    diameter=(9, 9, 9),
    noise_level=20,
    seed=42,
    engines=("python", "numba", "cupy"),
    ref_engine="python",
    tol=1.0,
)
df_detect_iso.sort_values("engine")

In [ ]:
# Optional: anisotropic 3D runtime benchmark
df_runtime_aniso = benchmark_all(
    shape=(128, 128, 64),
    count=200,
    diameter=(7, 9, 9),
    noise_level=20,
    seed=42,
    engines=("python", "numba", "cupy"),
    warmup=2,
    repeats=5,
)
df_runtime_aniso.sort_values("engine")

In [ ]:
# Optional: anisotropic 3D detection comparison
df_detect_aniso = compare_detection_all(
    shape=(128, 128, 64),
    count=200,
    diameter=(7, 9, 9),
    noise_level=20,
    seed=42,
    engines=("python", "numba", "cupy"),
    ref_engine="python",
    tol=1.0,
)
df_detect_aniso.sort_values("engine")

## Notes
- `cupy` が失敗する場合は、GPUランタイム有効化と `cupy-cuda12x` の導入状態を確認。
- PyPI版 `trackpy` を入れた場合、`engine='cupy'` が未実装の版ではエラーになります。
- このブランチの変更を測る場合は `/content/trackpy-master` をアップロードして実行してください。
- 検出比較は `python` を基準に最近傍マッチ（`tol=1.0` px）で集計しています。
